In [ ]:
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

output_dir = Path('/kaggle/working/results')
output_dir.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Using device: {device}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
optimizer = 'soap'
steps = 50
eval_every = 1
schedule_steps = 2000
seed = 0
K = 1
n_pde = 1000
n_boundary_each = 100
lr = 1e-3
betas = (0.95, 0.95)
weight_decay = 0.0
precondition_frequency = 10

dataset_dir = Path('/kaggle/input/datasets/priyansh2006/soap-pinn')
schedule_path = dataset_dir / 'poisson_seed0_K1_schedule (1).pt'
soap_source_path = dataset_dir / 'soap.py'
output_name = 'stage2_soap_official_seed0_K1_debug50.json'

print({
    'optimizer': optimizer,
    'steps': steps,
    'eval_every': eval_every,
    'seed': seed,
    'K': K,
    'n_pde': n_pde,
    'n_boundary_each': n_boundary_each,
    'lr': lr,
    'betas': betas,
    'weight_decay': weight_decay,
    'precondition_frequency': precondition_frequency,
    'schedule': str(schedule_path),
    'soap_source': str(soap_source_path),
    'output': str(output_dir / output_name),
})

In [ ]:
class PINN(nn.Module):
    def __init__(self, width=128, hidden_layers=4):
        super().__init__()
        layers = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, xy):
        return self.net(xy)


def set_seed(value):
    torch.manual_seed(value)
    np.random.seed(value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(value)

In [ ]:
def poisson_residual(model, xy):
    prediction = model(xy)
    first = torch.autograd.grad(
        prediction, xy, torch.ones_like(prediction), create_graph=True, retain_graph=True
    )[0]
    u_xx = torch.autograd.grad(
        first[:, 0:1], xy, torch.ones_like(first[:, 0:1]), create_graph=True, retain_graph=True
    )[0][:, 0:1]
    u_yy = torch.autograd.grad(
        first[:, 1:2], xy, torch.ones_like(first[:, 1:2]), create_graph=True, retain_graph=True
    )[0][:, 1:2]
    forcing = 2.0 * math.pi ** 2 * torch.sin(math.pi * xy[:, 0:1]) * torch.sin(math.pi * xy[:, 1:2])
    return -(u_xx + u_yy) - forcing


def relative_l2(model):
    grid = torch.linspace(0.0, 1.0, 81, device=device)
    xx, yy = torch.meshgrid(grid, grid, indexing='ij')
    xy = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    with torch.no_grad():
        prediction = model(xy).reshape(-1)
    reference = torch.sin(math.pi * xy[:, 0]) * torch.sin(math.pi * xy[:, 1])
    return float(torch.linalg.vector_norm(prediction - reference) / (torch.linalg.vector_norm(reference) + 1e-12))


def flat_grad(model):
    gradients = [parameter.grad.detach().reshape(-1) for parameter in model.parameters() if parameter.grad is not None]
    if not gradients:
        raise RuntimeError('No gradients available')
    return torch.cat(gradients)

In [ ]:
if not schedule_path.exists():
    raise FileNotFoundError(
        f'Frozen schedule not found at {schedule_path}. Upload poisson_seed0_K1_schedule.pt first.'
    )
schedule = torch.load(schedule_path, map_location=device, weights_only=True)
print('Schedule length:', len(schedule))
if len(schedule) != schedule_steps:
    raise ValueError(f'Expected schedule length {schedule_steps}, found {len(schedule)}')

In [ ]:
import sys
import inspect

if not soap_source_path.exists():
    raise FileNotFoundError(f'Repository soap.py not found at {soap_source_path}')
sys.path.insert(0, str(soap_source_path.parent))
from soap import SOAP

soap_source = inspect.getsourcefile(SOAP)
print('SOAP source:', soap_source)
if Path(soap_source).resolve() != soap_source_path.resolve():
    raise RuntimeError(f'Unexpected SOAP source: {soap_source}')

In [ ]:
set_seed(seed)
model = PINN().to(device)
optimizer_instance = SOAP(
    model.parameters(),
    lr=lr,
    betas=betas,
    weight_decay=weight_decay,
    precondition_frequency=precondition_frequency,
)
print('Model and official SOAP optimizer initialized.')

In [ ]:
history = []
previous_gradient = None
start_time = time.perf_counter()
model.train()

for step in range(1, steps + 1):
    current_pde = schedule[step - 1]['pde'].detach().clone().requires_grad_(True)
    current_boundary = schedule[step - 1]['bc']
    optimizer_instance.zero_grad(set_to_none=True)

    loss_pde = torch.mean(poisson_residual(model, current_pde).square())
    loss_boundary = torch.mean(model(current_boundary).square())
    loss = loss_pde + loss_boundary
    if not torch.isfinite(loss):
        print(f'NON-FINITE LOSS AT STEP {step}')
        break
    loss.backward()

    gradient = flat_grad(model)
    if not torch.isfinite(gradient).all():
        print(f'NON-FINITE GRADIENT BEFORE STEP {step}')
        break
    gradient_norm = torch.linalg.vector_norm(gradient).item()
    gradient_cosine = float('nan') if previous_gradient is None else float(
        torch.dot(gradient, previous_gradient)
        / (torch.linalg.vector_norm(gradient) * torch.linalg.vector_norm(previous_gradient) + 1e-12)
    )
    previous_gradient = gradient.clone()

    parameters_before = torch.cat([
        parameter.detach().reshape(-1)
        for parameter in model.parameters()
    ])
    optimizer_instance.step()
    parameters_after = torch.cat([
        parameter.detach().reshape(-1)
        for parameter in model.parameters()
    ])
    update_norm = torch.linalg.vector_norm(parameters_after - parameters_before).item()

    if not torch.isfinite(parameters_after).all():
        print(f'NON-FINITE PARAMETERS AFTER STEP {step}')
        print('update_norm =', update_norm)
        break

    row = {
        'step': step,
        'loss': loss.item(),
        'loss_pde': loss_pde.item(),
        'loss_bc': loss_boundary.item(),
        'rel_l2': relative_l2(model),
        'grad_norm': gradient_norm,
        'grad_cos_prev': gradient_cosine,
        'param_update_norm': update_norm,
        'effective_update': step > 1,
        'elapsed_sec': time.perf_counter() - start_time,
    }
    history.append(row)
    print(
        f'step={step:3d} '
        f'loss={loss.item():.6e} '
        f'grad_norm={gradient_norm:.6e} '
        f'update_norm={update_norm:.6e}'
    )

In [ ]:
result = {
    'stage': 2,
    'optimizer': optimizer,
    'implementation': 'repository soap.py',
    'soap_source': inspect.getsourcefile(SOAP),
    'seed': seed,
    'K': K,
    'n_pde': n_pde,
    'n_boundary_each': n_boundary_each,
    'lr': lr,
    'betas': list(betas),
    'weight_decay': weight_decay,
    'precondition_frequency': precondition_frequency,
    'steps': steps,
    'eval_every': eval_every,
    'schedule_steps': schedule_steps,
    'schedule_file': schedule_path.name,
    'device': str(device),
    'torch_version': torch.__version__,
    'history': history,
}
result_path = output_dir / output_name
result_path.write_text(json.dumps(result, indent=2))
print(f'Saved: {result_path}')

In [ ]:
print(f'Result file: {result_path}')
print(f'SOAP source: {result["soap_source"]}')
print(f'Completed diagnostic steps: {len(result["history"])}')
if result['history']:
    print(f'Final metrics: {result["history"][-1]}')